In [ ]:
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack
from sklearn.metrics import accuracy_score

In [ ]:
import nltk 
nltk.download('stopwords')

In [ ]:
train_df=pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df=pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
train_df.isna().sum()

In [ ]:
test_df.isna().sum()

## A. Dropping Location feature

In [ ]:
new_df=train_df.drop(columns=['location'])
test_new_df=test_df.drop(columns=['location'])

In [ ]:
test_df['keyword'].value_counts().head(60)

In [ ]:
new_df.isna().sum()

In [ ]:
new_df.dropna(inplace=True)
mode=test_new_df['keyword'].mode()[0]
test_new_df['keyword'].fillna(mode, inplace=True)

In [ ]:
new_df.isna().sum()

In [ ]:
test_new_df.isna().sum()

In [ ]:
porterstemmer=PorterStemmer() 

In [ ]:
def stemming(content):
    stemmed=re.sub('[^a-zA-Z]', ' ', content)
    stemmed=stemmed.lower()
    stemmed=stemmed.split()
    stemmed=[porterstemmer.stem(word) for word in stemmed if not word in stopwords.words('english')]
    stemmed=' '.join(stemmed)
    return stemmed

In [ ]:
new_df['text']=new_df['text'].apply(stemming)
test_new_df['text']=test_new_df['text'].apply(stemming)

In [ ]:
new_df['keyword']=new_df['keyword'].apply(stemming)
test_new_df['keyword']=test_new_df['keyword'].apply(stemming)

In [ ]:
vectorizer1=TfidfVectorizer()
vectorizer1.fit(new_df['text'])
texttrain=vectorizer1.transform(new_df['text'])
texttest=vectorizer1.transform(test_new_df['text'])


In [ ]:
vectorizer2=TfidfVectorizer()
vectorizer2.fit(new_df['keyword'])
keywordtrain=vectorizer2.transform(new_df['keyword'])
keywordtest=vectorizer2.transform(test_new_df['keyword'])


In [ ]:
Xtrain = hstack([texttrain, keywordtrain])
Xtest = hstack([texttest, keywordtest])

In [ ]:
# from sklearn.preprocessing import StandardScaler
# st=StandardScaler(with_mean=False)
# st.fit(Xtrain)
# Xtrain=st.transform(Xtrain)
# Xtest=st.transform(Xtest)

In [ ]:
X_train, X_test, y_train, y_test=train_test_split(Xtrain, new_df['target'], test_size=0.2, random_state=2)

In [ ]:
model1=LogisticRegression(C=2.75)

In [ ]:
model1.fit(X_train,y_train)

In [ ]:
y_pred1=model1.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred1)

In [ ]:
y_pred_test=model1.predict(Xtest)


In [ ]:
submission = pd.DataFrame({
    'id': test_new_df['id'],
    'target': y_pred_test
})

submission.to_csv('submission.csv', index=False)